# Energy Landscape Visualization for Ring Attractor Networks

This notebook visualizes the energy landscape of a ring attractor network based on the energy function:

$$E(V) = \sum_{i=1}^{N}\left[\frac{1}{2}V_i^2 - (V_{\rm rest}+I_i^{\rm ext})\,V_i - \,V_i\sum_{j=1}^{N}\,W_{ij}\,\phi(V_j)\ - \,W_{ii}\Phi(V_j) + \text{constant}\right]$$

where:
- $V_i$ is the membrane potential of neuron $i$
- $V_{\rm rest}$ is the resting potential
- $I_i^{\rm ext}$ is the external input to neuron $i$
- $W_{ij}$ is the synaptic weight from neuron $j$ to neuron $i$
- $\phi(V_j)$ is a nonlinear function of the membrane potential of neuron $j$
- $\Phi(V_j)$ is the antiderivative of $\phi(V_j)$
- $N$ is the number of neurons in the network


Note that the equation can be approximated as follows (under certain conditions):
$$E(V) = \sum_{i=1}^{N}\left[\frac{1}{2}V_i^2 - (V_{\rm rest}+I_i^{\rm ext})\,V_i\right] - \frac{1}{2}\sum_{i,j=1}^{N}W_{ij}\,\phi(V_i)\,\phi(V_j) + \text{constant}$$

We'll create 3D visualizations to understand how the network dynamics are shaped by this energy function.

In [1]:
from brian2 import *
import sys
sys.path.append('Neuron and Synapse Models')
sys.path.append('Tools')

from neuronModels import *
from ringAttractorTEMP import *
from plottingTools import *
from utils import *

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from scipy.ndimage import gaussian_filter1d, minimum_filter
import plotly.graph_objects as go
import plotly.express as px
from scipy.interpolate import griddata
from scipy.optimize import minimize
from ipywidgets import interact, FloatSlider, fixed
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Set up simulation parameters
defaultclock.dt = 0.1*ms

ModuleNotFoundError: No module named 'ringAttractorTEMP'

## Define the Activation Function and its Integral

We'll use the `rect_power` function from the `utils` module as our activation function $\phi(V)$ and `rect_powerInt` as its integral $\Phi(V)$ for the energy calculation.

## Create the Ring Attractor Network

We'll create a ring attractor network using the same parameters as in the parameter tuning notebook.

In [ ]:
def create_ring_attractor(sigma_exc=1.0, sigma_inh=2.0, g_exc=4.0*mV, g_inh=-4.0*mV, stimulus_center=0):
    # Basic parameters
    tau = 10 * ms
    sigma_noise = 1 * mV
    V_rest = -70 * mV
    I0 = 30 * mV
    num_neurons = 120
    stimulus_width = 0.5
    
    # Define neuron positions
    positions = linspace(0, 2*pi, num_neurons, endpoint=False)
    
    # Calculate external input
    d = np.angle(np.exp(1j * (positions - stimulus_center)))
    I_ext_array = I0 * np.exp(-(d**2) / (2 * stimulus_width**2))
    
    # Set up neuron model
    neuron_eq = Equations(LIF_xi_eq, tau=tau, V_rest=V_rest, sigma_noise=sigma_noise)
    
    # Set up ring attractor
    Vth = -48 * mV
    V_reset = -80 * mV
    refractory_period = 5 * ms
    syn_profile = 'mexican_hat'
    
    # Create the ring attractor network
    ringAttractor = RingAttractor(neuron_eq, 
                            num_neurons, 
                            Vth, V_reset, refractory_period,
                            syn_profile=syn_profile,
                            autapse=True,
                            sigma_exc=sigma_exc,
                            sigma_inh=sigma_inh, 
                            g_exc=g_exc, 
                            g_inh=g_inh)
    
    # Set external input
    ringAttractor.ring_pool.I_ext = I_ext_array
    
    return ringAttractor, positions, I_ext_array, num_neurons

In [ ]:
# Create the ring attractor with default parameters
sigma_exc = 1.0
sigma_inh = 2.0
g_exc = 4.0 * mV
g_inh = -4.0 * mV
stimulus_center = 0
Vth = -48 * mV
V_rest = -70 * mV

# Initialize the network
ringAttractor, positions, I_ext, num_neurons = create_ring_attractor(
    sigma_exc=sigma_exc, 
    sigma_inh=sigma_inh, 
    g_exc=g_exc, 
    g_inh=g_inh, 
    stimulus_center=stimulus_center
)

WARNING    The object 'ring_synapses' is getting deleted, but was never included in a network. This probably means that you did not store the object reference in a variable, or that the variable was not used to construct the network.
The object was created here (most recent call only):
  File '/home/bmaacaron-iit.local/Documents/Git Repos/JointAttractorNets/Neuron and Synapse Models/ringAttractorTEMP.py', line 95, in __init__
    self.ring_synapses = Synapses(self.ring_pool, self.ring_pool, model='w : volt', [brian2.core.base.unused_brian_object]


## Calculate Energy Function

Define a function to calculate the energy for a given network state using the formula:

$$E(V) = \sum_{i=1}^{N}\left[\frac{1}{2}V_i^2 - (V_{\rm rest}+I_i^{\rm ext})\,V_i - \,V_i\sum_{j=1}^{N}\,W_{ij}\,\phi(V_j)\ - \,W_{ii}\Phi(V_j) + \text{constant}\right]$$

In [ ]:
def calculate_energy(V, ringAttractor, V_rest, I_ext, Vth=-48*mV):
    """Calculate the energy of the network for a given state V"""
    # Get the raw weight matrix
    W_raw = ringAttractor.ring_synapses.w_[:]  # Raw weights
            
    # Reshape the weight matrix if needed
    W = W_raw.reshape(num_neurons, num_neurons)

    # Convert membrane potentials to unitless values
    V_unitless = V / mV  # Convert to unitless values (as float numbers)
    V_rest_unitless = V_rest / mV
    I_ext_unitless = I_ext / mV
    Vth_unitless = float(Vth / mV)  # Convert threshold to unitless
    
    # First term: intrinsic energy (using unitless values)
    intrinsic_energy = 0.5 * np.sum(V_unitless**2) - np.sum((V_rest_unitless + I_ext_unitless) * V_unitless)
    
    # Calculate phi(V) for all neurons (using rect_power with unitless values)
    phi_V = rect_power(V_unitless, a=3.3, V0=Vth_unitless, p=1.0)
    
    # Second term: V_i interaction with sum_j(W_ij*phi(V_j))
    W_phi = W @ phi_V  # This gives sum_j(W_ij*phi(V_j)) for each i
    interaction_term = -np.sum(V_unitless * W_phi)
    
    # Third term: antiderivative term with diagonal weights
    Phi_V = rect_powerInt(V_unitless, a=3.3, V0=Vth_unitless, p=1.0)
    diag_W = np.diag(W)  # Extract diagonal elements of W
    antiderivative_term = -np.sum(diag_W * Phi_V)
    
    # Total energy (constant term omitted as it doesn't affect dynamics)
    total_energy = intrinsic_energy + interaction_term + antiderivative_term
    
    return total_energy

CHeck out Marimo

In [ ]:
import itertools

PS_WEIGHT_EXC_S_N_c1_coarse_range = [5, 6, 7]           # NMDA weight coarse
PS_WEIGHT_EXC_S_N_c1_fine_range = [30, 80, 130]         # NMDA weight fine
NPDPIE_THR_S_P_c1_coarse_range = [3, 4, 5]              # NMDA gain coarse
NPDPIE_THR_S_P_c1_fine_range = [50, 120, 200]           # NMDA gain fine

# Core 3 parameters (ring populations)
PS_WEIGHT_EXC_F_N_c3_coarse_range = [5, 6, 7]           # AMPA weight coarse
PS_WEIGHT_EXC_F_N_c3_fine_range = [20, 80, 150]         # AMPA weight fine
PS_WEIGHT_EXC_S_N_c3_coarse_range = [5, 6, 7]           # NMDA weight coarse
PS_WEIGHT_EXC_S_N_c3_fine_range = [40, 100, 180]        # NMDA weight fine
PS_WEIGHT_INH_S_N_c3_coarse_range = [5, 6, 7]           # GABA B weight coarse
PS_WEIGHT_INH_S_N_c3_fine_range = [50, 120, 200]        # GABA B weight fine
NPDPIE_THR_F_P_c3_coarse_range = [3, 4, 5]              # AMPA gain coarse
NPDPIE_THR_F_P_c3_fine_range = [50, 120, 190]           # AMPA gain fine
NPDPIE_THR_S_P_c3_coarse_range = [3, 4, 5]              # NMDA gain coarse
NPDPIE_THR_S_P_c3_fine_range = [50, 120, 190]           # NMDA gain fine
NPDPII_THR_S_P_c3_coarse_range = [3, 4, 5]              # GABA B gain coarse
NPDPII_THR_S_P_c3_fine_range = [50, 120, 190]           # GABA B gain fine

# Create all parameter combinations
param_ranges = [
PS_WEIGHT_EXC_S_N_c1_coarse_range,   # params_tuple[0]
PS_WEIGHT_EXC_S_N_c1_fine_range,     # params_tuple[1]
NPDPIE_THR_S_P_c1_coarse_range,      # params_tuple[2]
NPDPIE_THR_S_P_c1_fine_range,        # params_tuple[3]
PS_WEIGHT_EXC_F_N_c3_coarse_range,   # params_tuple[4]
PS_WEIGHT_EXC_F_N_c3_fine_range,     # params_tuple[5]
PS_WEIGHT_EXC_S_N_c3_coarse_range,   # params_tuple[6]
PS_WEIGHT_EXC_S_N_c3_fine_range,     # params_tuple[7]
PS_WEIGHT_INH_S_N_c3_coarse_range,   # params_tuple[8]
PS_WEIGHT_INH_S_N_c3_fine_range,     # params_tuple[9]
NPDPIE_THR_F_P_c3_coarse_range,      # params_tuple[10]
NPDPIE_THR_F_P_c3_fine_range,        # params_tuple[11]
NPDPIE_THR_S_P_c3_coarse_range,      # params_tuple[12]
NPDPIE_THR_S_P_c3_fine_range,        # params_tuple[13]
NPDPII_THR_S_P_c3_coarse_range,      # params_tuple[14]
NPDPII_THR_S_P_c3_fine_range         # params_tuple[15]
]

# Generate all combinations of parameters
param_combinations = list(itertools.product(*param_ranges))
print(len(param_combinations), "parameter combinations generated.")

43046721 parameter combinations generated.
